<a href="https://colab.research.google.com/github/peter-cheun/FINTECH-P2Example/blob/main/FINA4075_P2_StrategyLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
# Cell 1 — mount Drive (Rule 1)
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
# Cell 2 — create the course folder
import os
P0 = "/content/drive/MyDrive/FINA4075/P2"
os.makedirs(P2, exist_ok=True)
print("Folder ready:", P2)

Folder ready: /content/drive/MyDrive/FINA4075/P2


In [62]:
# Cell 3 — pinned setup (Rule 2; -q omitted in P0 on purpose)
# Official pinned line (Fall 2026), as posted in D2L (Rule 2).
%pip install yfinance==0.2.66 sec-edgar-downloader==5.1.0

In [63]:
# Cell 4 — environment stamp (Rule 0)
import sys, pandas as pd, numpy as np, matplotlib, yfinance as yf
print(sys.version)
print("pandas", pd.__version__, "| numpy", np.__version__,
      "| matplotlib", matplotlib.__version__, "| yfinance", yf.__version__)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
pandas 2.2.3 | numpy 2.1.3 | matplotlib 3.10.0 | yfinance 0.2.66


In [51]:
# Cell 5 — hash check and load
import hashlib
sha = hashlib.sha256(open("panel_returns_2026-08-31.csv", "rb").read()).hexdigest()
print("SHA-256:", sha)
assert sha == PANEL_SHA256, "panel file does not match the Data Dictionary hash"

panel = pd.read_csv(PANEL_FILE, index_col=0, parse_dates=True)
info = pd.read_csv(TICKER_FILE).set_index("Ticker")
print("panel shape:", panel.shape, "| first month:", panel.index[0].date(), "| last month:", panel.index[-1].date())
print("missing values:", int(panel.isna().sum().sum()), "| tickers match the company list:", list(panel.columns) == list(info.index))

FileNotFoundError: [Errno 2] No such file or directory: 'panel_returns_2026-08-31.csv'

In [76]:
%%writefile /content/drive/MyDrive/FINA4075/P2/strategy.py
"""
strategy.py — LOCKED STRATEGY CODE (Track 1)   ·   FINTECH-P2   ·   FINA 4075/5075, Fall 2026

Strategy: Low volatility, 36 months (Strategy Menu, Section 2, item (c)).
strategy_rules.md governs if the two ever disagree.

Rule:
  For the portfolio held in month t, compute each stock's sample standard deviation (ddof = 1) of its
  raw simple monthly decimal returns over rows t-36 .. t-1 inclusive (36 observations).  A stock with
  any missing value in that window gets volatility +Inf.  Rank ascending (lowest volatility first),
  ties broken by ticker in ASCII order.  Hold the 20 lowest, 5% each, rebalanced at every month-end
  from data through the prior month-end only.  10 bp per side.

  Row 0 = January 2019.  First holding month = row 36 (January 2022, window rows 0..35).
  In-sample backtest: rows 36..91 (January 2022 - August 2026, 56 months).
  Live window: rows 92, 93, 94 (September, October, November 2026).

CODE RULE: this file is read-only after the 9/20/2026 pre-registration commit.  At each checkpoint
the panel is extended by one month (instructor's extension file) and this code is rerun unchanged:

    python strategy.py                                   # entry 0 (released panel only)
    python strategy.py --ext ext_2026-09.csv             # checkpoint 1
    python strategy.py --ext ext_2026-09.csv ext_2026-10.csv --delistings delistings.csv
"""
import argparse
import hashlib

import numpy as np
import pandas as pd

# ---------------------------------------------------------------- fixed parameters (rules file)
PANEL_SHA256 = "c9d3340eba3a3301a63bb788c87f1d6e9e55de58a3fec16b12a34b313cac076b"
PER_SIDE = 0.0010              # transaction cost, 10 basis points per side
N_HOLD = 20                    # exactly 20 holdings
WEIGHT = 1.0 / N_HOLD          # 0.05 each
WINDOW_START = 36              # window starts WINDOW_START months before month t  (t-36)
WINDOW_END = 1                 # window ends   WINDOW_END   months before month t  (t-1), inclusive
WINDOW_LEN = WINDOW_START - WINDOW_END + 1          # 36 observations
FIRST_TRADEABLE = WINDOW_START                      # row 36 = January 2022 (window rows 0..35)
LAST_IN_SAMPLE = 91                                 # row 91 = August 2026
IN_SAMPLE_MONTHS = LAST_IN_SAMPLE - FIRST_TRADEABLE + 1   # 56
DELIST_DEFAULT_RETURN = -1.0   # used when no terminal delisting return is supplied


# ---------------------------------------------------------------- data
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_panel(panel_path, ticker_path, extension_paths=(), verify_hash=True):
    """Read the course packet plus any extension files.  Returns (rets, dates, tickers, info).

    rets    : numpy array, T x N, simple monthly total returns (row 0 = January 2019).
              Extension rows may contain NaN for delisted names.
    dates   : the month-end dates of the rows
    tickers : list of the N ticker symbols, in the column order of the panel
    info    : the company list (ticker, company, sector, ...) indexed by ticker
    """
    if verify_hash:
        digest = sha256_of(panel_path)
        assert digest == PANEL_SHA256, f"panel SHA-256 mismatch: {digest}"

    panel = pd.read_csv(panel_path, index_col=0, parse_dates=True)
    info = pd.read_csv(ticker_path).set_index("Ticker")
    tickers = list(panel.columns)
    assert list(info.index) == tickers, "ticker list and panel columns differ"
    assert len(tickers) == 200, f"expected 200 stocks, found {len(tickers)}"
    assert not panel.isna().any().any(), "released panel has missing values"
    assert len(panel) == LAST_IN_SAMPLE + 1, f"released panel should have 92 rows, has {len(panel)}"

    for path in extension_paths:
        ext = pd.read_csv(path, index_col=0, parse_dates=True)
        extra = set(ext.columns) - set(tickers)
        assert not extra, f"{path}: unknown tickers {sorted(extra)}"
        ext = ext.reindex(columns=tickers)            # a missing column becomes NaN (delisted)
        ext = ext.apply(pd.to_numeric, errors="coerce")   # non-numeric entries count as missing
        panel = pd.concat([panel, ext])

    assert panel.index.is_monotonic_increasing and panel.index.is_unique, "dates out of order"
    return panel.to_numpy(dtype=float), panel.index, tickers, info


def load_delistings(path, dates, tickers):
    """Optional CSV with columns Ticker, Month (e.g. 2026-10).  Returns {column index: row index}
    where the row is the month during which the stock delisted.  Its terminal return (if the
    instructor supplies one) is expected in that row of the extension file."""
    if path is None:
        return {}
    df = pd.read_csv(path)
    periods = pd.PeriodIndex(dates, freq="M")
    col = {tk: i for i, tk in enumerate(tickers)}
    out = {}
    for tk, month in zip(df["Ticker"], df["Month"]):
        row = periods.get_loc(pd.Period(str(month), freq="M"))
        out[col[tk]] = int(row)
    return out


def active_at_close(t, n, delist_rows):
    """Boolean mask: stock is active at the close of month t-1 (has not delisted in any month <= t-1)."""
    mask = np.ones(n, dtype=bool)
    for i, row in (delist_rows or {}).items():
        if row <= t - 1:
            mask[i] = False
    return mask


def realized(r):
    """Month-t returns with missing values replaced by the -100% default (never dropped)."""
    r = np.array(r, dtype=float)
    r[~np.isfinite(r)] = DELIST_DEFAULT_RETURN
    return r


# ---------------------------------------------------------------- signal and selection
def signal(t, rets):
    """Sample standard deviation (ddof = 1) of raw simple returns over rows t-36 .. t-1 inclusive.

    Python slicing excludes the stop index, so rows t-36 .. t-1 are rets[t-36 : t].
    A stock without exactly 36 finite observations in the window gets +Inf.
    """
    assert t >= FIRST_TRADEABLE, f"row {t} has no complete {WINDOW_LEN}-month window"
    window = rets[t - WINDOW_START: t - WINDOW_END + 1]
    assert window.shape[0] == WINDOW_LEN == 36
    valid = np.isfinite(window).all(axis=0)
    vol = np.full(window.shape[1], np.inf)
    if valid.any():
        vol[valid] = window[:, valid].std(axis=0, ddof=1)
    return vol


def select_holdings(t, rets, tickers, delist_rows=None):
    """The 20 stocks held during month t, in rank order (list of column indices).

    Lowest volatility first; exact float64 ties broken by ASCII ticker order (Python str order).
    If fewer than 20 active stocks have finite volatility, the remaining slots are filled with the
    remaining active stocks in alphabetical ticker order.
    """
    vol = signal(t, rets)
    active = active_at_close(t, len(tickers), delist_rows)
    cand = [i for i in range(len(tickers)) if active[i]]
    ranked = sorted((i for i in cand if np.isfinite(vol[i])), key=lambda i: (vol[i], tickers[i]))
    filler = sorted((i for i in cand if not np.isfinite(vol[i])), key=lambda i: tickers[i])
    chosen = (ranked + filler)[:N_HOLD]
    assert len(chosen) == N_HOLD, f"row {t}: only {len(chosen)} active stocks available"
    return chosen


# ---------------------------------------------------------------- paper trading
def run_track(select_fn, rets, tickers, t0, t1, delist_rows=None, per_side=PER_SIDE):
    """Paper-trade a selection rule from month t0 to month t1 (row indices, inclusive).

    Equal weights (1/20); one-way turnover = names not held last month / 20 (the first month of the
    run counts as a full purchase); monthly cost = 2 x per_side x turnover; a held stock with no
    return for month t is scored at -100%; growth path starts at 1.0.
    Returns (values, turnovers, holdings) with len(values) == months + 1; holdings are ticker lists.
    """
    prev, vals, turns, holdings = set(), [1.0], [], []
    for t in range(t0, t1 + 1):
        chosen = select_fn(t, rets, tickers, delist_rows)
        hold = set(chosen)
        turn = 1.0 if not prev else len(hold - prev) / N_HOLD
        gross = WEIGHT * realized(rets[t, sorted(hold)]).sum()      # (1/20) * sum of 20 returns
        vals.append(vals[-1] * (1 + gross - 2 * per_side * turn))
        prev = hold
        turns.append(turn)
        holdings.append([tickers[i] for i in chosen])
    return np.array(vals), np.array(turns), holdings


def bench(rets, t0, t1, delist_rows=None):
    """Equal-weighted mean of all stocks active at the close of month t-1, rebalanced monthly,
    no costs.  A stock delisting during month t is included at its terminal return (-100% if
    none is supplied) and dropped from month t+1 on."""
    vals = [1.0]
    for t in range(t0, t1 + 1):
        active = active_at_close(t, rets.shape[1], delist_rows)
        vals.append(vals[-1] * (1 + realized(rets[t, active]).mean()))
    return np.array(vals)


def stats(vals, turns=None):
    """CAGR, annualized volatility, Sharpe (rf = 0, course convention CAGR / vol), maximum
    drawdown, growth of $10,000, and (for a strategy) annual one-way turnover and cost drag."""
    months = len(vals) - 1
    monthly = vals[1:] / vals[:-1] - 1
    cagr = vals[-1] ** (12 / months) - 1
    vol = monthly.std(ddof=1) * np.sqrt(12) if months > 1 else np.nan
    maxdd = (vals / np.maximum.accumulate(vals) - 1).min()
    out = {"CAGR": cagr, "Annualized volatility": vol, "Sharpe (rf = 0)": cagr / vol,
           "Maximum drawdown": maxdd, "Growth of $10,000": 10000 * vals[-1]}
    if turns is not None:
        out["Annual one-way turnover"] = turns.mean() * 12
        out["Cost drag per year"] = 2 * PER_SIDE * turns.mean() * 12
    return out


def print_stats(title, s):
    print(f"\n{title}")
    for k, v in s.items():
        print(f"  {k:<26}{v:>14,.2f}" if k == "Growth of $10,000" else f"  {k:<26}{v:>14.4f}")


# ---------------------------------------------------------------- checkpoint run
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--panel", default="panel_returns_2026-08-31.csv")
    ap.add_argument("--tickers", default="P2_ticker_list_2026-08-31.csv")
    ap.add_argument("--ext", nargs="*", default=[], help="extension files, in month order")
    ap.add_argument("--delistings", default=None, help="optional CSV: Ticker, Month (YYYY-MM)")
    args = ap.parse_args()

    rets, dates, tickers, info = load_panel(args.panel, args.tickers, args.ext)
    delist_rows = load_delistings(args.delistings, dates, tickers)
    T = rets.shape[0]

    # In-sample backtest: January 2022 - August 2026
    vals, turns, holdings = run_track(select_holdings, rets, tickers,
                                      FIRST_TRADEABLE, LAST_IN_SAMPLE, delist_rows)
    assert len(vals) - 1 == IN_SAMPLE_MONTHS == 56
    print(f"In-sample: {dates[FIRST_TRADEABLE]:%b %Y} - {dates[LAST_IN_SAMPLE]:%b %Y} "
          f"({IN_SAMPLE_MONTHS} months)")
    print_stats("Low volatility (36m), top 20, net of costs", stats(vals, turns))
    print_stats("Benchmark (equal-weight universe, no costs)",
                stats(bench(rets, FIRST_TRADEABLE, LAST_IN_SAMPLE, delist_rows)))

    # Live months scored so far (rows 92 .. T-1), month by month
    if T - 1 > LAST_IN_SAMPLE:
        lv, lt, lh = run_track(select_holdings, rets, tickers, LAST_IN_SAMPLE + 1, T - 1, delist_rows)
        bv = bench(rets, LAST_IN_SAMPLE + 1, T - 1, delist_rows)
        print("\nLive window, month by month (net strategy vs benchmark):")
        for k, t in enumerate(range(LAST_IN_SAMPLE + 1, T)):
            print(f"  {dates[t]:%b %Y}  strategy {lv[k+1]/lv[k]-1:+.4%}  "
                  f"benchmark {bv[k+1]/bv[k]-1:+.4%}  turnover {lt[k]:.2f}")

    # Portfolio for the next month (row T), selected from rows T-36 .. T-1 -> monitoring_log.md
    next_month = (pd.Period(dates[-1], freq="M") + 1).strftime("%B %Y")
    vol = signal(T, rets)
    chosen = select_holdings(T, rets, tickers, delist_rows)
    print(f"\nPortfolio for {next_month} (row {T}), formed from rows {T-36}-{T-1} "
          f"({dates[T-36]:%b %Y} - {dates[T-1]:%b %Y}):")
    for rank, i in enumerate(chosen, 1):
        print(f"  {rank:>2}. {tickers[i]:<8} vol {vol[i]:.6f}  weight {WEIGHT:.2f}")


if __name__ == "__main__":
    main()


def bench(rets, t0, t1):
    """Equal-weighted average of all 200 stocks, rebalanced monthly, no costs."""
    vals = [1.0]
    for t in range(t0, t1 + 1):
        vals.append(vals[-1] * (1 + rets[t].mean()))
    return np.array(vals)


def stats(vals, turns=None):
    """CAGR, annualized volatility, Sharpe (rf = 0), maximum drawdown, annual one-way turnover."""
    months = len(vals) - 1
    monthly = vals[1:] / vals[:-1] - 1
    cagr = vals[-1] ** (12 / months) - 1
    vol = rets[t-36:t, :].std(axis=0, ddof=1)
    maxdd = (vals / np.maximum.accumulate(vals) - 1).min()
    out = {"CAGR": cagr, "Annualized volatility": vol, "Sharpe (rf = 0)": cagr / vol,
           "Maximum drawdown": maxdd, "Growth of $10,000": 10000 * vals[-1]}
    if turns is not None:
        out["Annual one-way turnover"] = turns.mean() * 12
        out["Cost drag per year"] = 2 * PER_SIDE * turns.mean() * 12
    return out

Writing /content/drive/MyDrive/FINA4075/P2/strategy.py


In [80]:
# Cell 7 — import the locked code from Drive and bind the rule to this panel
import importlib, sys
if P2 not in sys.path:
    sys.path.insert(0, P2)
import strategy as S
importlib.reload(S)

rets, dates, tickers, info = S.load_panel(PANEL_FILE, TICKER_FILE)
def rule(t, rets):
    return S.select_holdings(t, rets, tickers)
print("rows (months):", len(dates), "| first tradeable row:", S.FIRST_TRADEABLE, "=", dates[S.FIRST_TRADEABLE].date())

NameError: name 'PANEL_FILE' is not defined

In [82]:
# Cell 8 — run both tracks' in-sample paths and the stats table
t0, t1 = S.FIRST_TRADEABLE, len(dates) - 1                 # row 12 (Jan 2020) .. row 91 (Aug 2026)
vals, turns, holdings = S.run_track(rule, rets, t0, t1)
bvals = S.bench(rets, t0, t1)

strat_stats, bench_stats = S.stats(vals, turns), S.stats(bvals)
rows = ["CAGR", "Annualized volatility", "Sharpe (rf = 0)", "Maximum drawdown", "Annual one-way turnover", "Cost drag per year", "Growth of $10,000"]
table = pd.DataFrame({"Intermediate momentum (net)": [strat_stats.get(r) for r in rows],
                      "EW benchmark (no costs)": [bench_stats.get(r) for r in rows]}, index=rows)
def fmt(v, r):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "—"
    if r == "Growth of $10,000": return f"${v:,.0f}"
    if r == "Sharpe (rf = 0)": return f"{v:.2f}"
    return f"{v*100:.1f}%"
shown = pd.DataFrame({c: [fmt(table.loc[r, c], r) for r in rows] for c in table.columns}, index=rows)
print(f"In-sample window: {dates[t0].date()} to {dates[t1].date()} ({t1 - t0 + 1} months)")
print(shown.to_string())
table.to_csv(OUT + "/insample_stats_2026-08-31.csv")

NameError: name 'dates' is not defined

In [83]:
# Cell 9 — growth-of-$10,000 exhibit (saved to Drive, Rule 7)
import matplotlib.pyplot as plt
x = dates[t0 - 1 : t1 + 1]                                 # month-ends; the path starts one month before the first holding month
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(x, 10000 * vals, label="Track 1: intermediate momentum (t−12…t−7), top 20 EW, net of 10 bp/side", lw=1.8)
ax.plot(x, 10000 * bvals, label="Equal-weighted panel (200 stocks), no costs", lw=1.8)
ax.axhline(10000, ls="--", lw=0.8, color="gray")
ax.set_title("Growth of $10,000 — in-sample, January 2020 to August 2026 (80 months)")
ax.set_ylabel("Portfolio value ($)")
ax.legend(loc="upper left", fontsize=9)
ax.text(0.01, -0.16, "Source: panel_returns_2026-08-31.csv (Yahoo Finance, dividends reinvested). Panel has survivorship and selection bias; rf = 0 for Sharpe.",
        transform=ax.transAxes, fontsize=7.5, color="gray")
plt.tight_layout()
fig.savefig(OUT + "/insample_growth_2026-08-31.png", dpi=150, bbox_inches="tight")
plt.show()

NameError: name 'dates' is not defined

In [84]:
# Cell 10 — one month by hand: the last in-sample month, August 2026 (row 91)
t = t1
hold_now, hold_prev = holdings[t - t0], holdings[t - t0 - 1]
names = sorted(tickers[i] for i in hold_now)
r_month = pd.Series({tickers[i]: rets[t, i] for i in hold_now}).sort_index()
gross = r_month.mean()
turn = len(hold_now - hold_prev) / S.N_HOLD
cost = 2 * S.PER_SIDE * turn
print(f"Month scored: {dates[t].date()} | signal window: {dates[t-12].date()} .. {dates[t-7].date()}")
print("Holdings (20):", ", ".join(names))
print((100 * r_month).round(2).to_string())
print(f"\ngross = mean of the 20 returns = {gross:.6f} ({gross*100:.3f}%)")
print(f"turnover = names replaced / 20 = {len(hold_now - hold_prev)} / 20 = {turn:.2f}  ->  cost = 2 x 0.0010 x {turn:.2f} = {cost:.6f}")
print(f"net = {gross - cost:.6f} ({(gross - cost)*100:.3f}%)  ->  on
{10000 * (gross - cost):,.2f}")
print("Three holdings to check against the CSV by hand:", ", ".join(f"{n} {rets[t, tickers.index(n)]*100:+.3f}%" for n in names[:3]))
print("Engine's own value for this month:", f"{vals[t - t0 + 1] / vals[t - t0] - 1:.6f}", "(must equal net above)")

SyntaxError: unterminated f-string literal (detected at line 14) (760303980.py, line 14)

In [85]:
# Cell 11 — briefing table as of a month-end (default: the last row of the panel)
def briefing_table(panel, info, as_of=None):
    r = panel if as_of is None else panel.loc[:as_of]
    as_of = r.index[-1].date()
    r1 = r.iloc[-1]
    r3 = (1 + r.iloc[-3:]).prod() - 1
    r12 = (1 + r.iloc[-12:]).prod() - 1
    tbl = pd.DataFrame({"Ticker": r.columns,
                        "Company": info.loc[r.columns, "Company"].values,
                        "Sector": info.loc[r.columns, "Sector"].values,
                        "1M": (100 * r1.values).round(1) + 0.0, "3M": (100 * r3.values).round(1) + 0.0, "12M": (100 * r12.values).round(1) + 0.0})   # + 0.0 turns -0.0 into 0.0
    return tbl.sort_values("Ticker").reset_index(drop=True), as_of

def briefing_text(tbl, as_of):
    lines = [f"BRIEFING TABLE as of {as_of} (month-end). Trailing total returns in percent, dividends reinvested. {len(tbl)} stocks.",
             "Ticker | Company | Sector | 1M | 3M | 12M"]
    for row in tbl.itertuples(index=False):
        lines.append(f"{row.Ticker} | {row.Company} | {row.Sector} | {row[3]:+.1f} | {row[4]:+.1f} | {row[5]:+.1f}")
    return "\n".join(lines)

tbl, as_of = briefing_table(panel, info)                 # August 2026 month-end for the 9/20 commit
text = briefing_text(tbl, as_of)
open(OUT + f"/briefing_table_{as_of}.txt", "w").write(text)
tbl.to_csv(OUT + f"/briefing_table_{as_of}.csv", index=False)
print(text)

NameError: name 'panel' is not defined

In [86]:
# Cell 12 — the locked rule's September 2026 portfolio, from data through August 2026
t_next = len(dates)                                        # row 92 = September 2026
hold_next = rule(t_next, rets)
sig_next = S.signal(t_next, rets)
sep = sorted(tickers[i] for i in hold_next)
print(f"Signal window: {dates[t_next-12].date()} .. {dates[t_next-7].date()} (six months)")
print("TRACK 1 September 2026 holdings (alphabetical):")
print(", ".join(sep))
ranked = pd.Series(sig_next, index=tickers).sort_values(ascending=False)
print("\nRanks 18-23 around the boundary (signal = compounded 6-month return):")
print((100 * ranked.iloc[17:23]).round(2).to_string())
open(OUT + "/track1_holdings_2026-09.txt", "w").write(", ".join(sep) + "\n")
print("\nSaved:", sorted(os.listdir(OUT)))

NameError: name 'dates' is not defined